# Reproduction of reported values

Runs the analysis scripts in order and prints every reported value.

**Requirements.** `data/dataset.xlsx` and `data/applicability_domain.csv` next to this notebook. Python with scikit-learn, numpy, scipy, pandas and openpyxl. Results were obtained with Python 3.13.18, scikit-learn 1.9.1, numpy 2.5.3, scipy 1.18.1, pandas 3.0.5.

**Notes.** A step is skipped when its result file already exists in `repro/`. A full run takes several hours on two processor cores. Fit times depend on the machine; all other values use fixed seeds.

## 1. Setup

In [ ]:
import os, sys, json, time, shutil, subprocess
import numpy as np

BASE = os.path.abspath('repro'); AN = os.path.join(BASE, 'an'); BN = os.path.join(BASE, 'bench'); DATA = os.path.join(AN, 'data')
for d in (os.path.join(AN, 'out'), os.path.join(BN, 'out'), DATA): os.makedirs(d, exist_ok=True)
for f in ('dataset.xlsx', 'applicability_domain.csv'): shutil.copy(os.path.join('data', f), os.path.join(DATA, f))

def run(args, cwd, outputs, log):
    if all(os.path.exists(os.path.join(cwd, o)) for o in outputs): print('done:', ', '.join(outputs)); return
    t = time.time(); print('running', ' '.join(args), flush=True)
    with open(os.path.join(cwd, log), 'w') as fh: r = subprocess.run([sys.executable] + args, cwd=cwd, stdout=fh, stderr=subprocess.STDOUT)
    assert r.returncode == 0, 'failed, see ' + os.path.join(cwd, log)
    print('  %.1f min' % ((time.time() - t) / 60))

def run_parallel(jobs, cwd):
    procs = []
    for args, outputs, log in jobs:
        if all(os.path.exists(os.path.join(cwd, o)) for o in outputs): print('done:', ', '.join(outputs)); continue
        print('running', ' '.join(args), flush=True); procs.append((subprocess.Popen([sys.executable] + args, cwd=cwd, stdout=open(os.path.join(cwd, log), 'w'), stderr=subprocess.STDOUT), log))
    t = time.time()
    for p, log in procs: assert p.wait() == 0, 'failed, see ' + os.path.join(cwd, log)
    if procs: print('  %.1f min' % ((time.time() - t) / 60))


## 2. Analysis scripts

In [ ]:
%%writefile repro/an/common.py
# Data loading, descriptor pipeline, Kennard-Stone split, model definition and metrics.
import copy, numpy as np, pandas as pd
from scipy.stats import boxcox, pearsonr, spearmanr, norm
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import LeaveOneOut
SEED=42
LOGF=['Half life_matrix_avg','tortuosity','interface_rot','matrix_CED','fiber_dia_or_average_particle_size_mm']
DROP=['hydrophyllic_gradient','matrix_sol_parameter','flory_huggins','logKOC_matrix_avg','sol_mismatch','logP_fiber','Lignin_Percentage','Cellulose_Percentage','Half life_fiber_avg','logKOC_fiber_avg','fiber_CED','fiber_rotatable_bond_count','Specific Interface Area']
PRE=[0,8,18]
MF=['Half life_matrix_avg','logP_matrix','matrix_CED']
FF=['Hemicellulose_Percentage','tortuosity','interface_rot','fiber_sol_parameter','fiber_dia_or_average_particle_size_mm']
TFO,THX='first_order_rate','hixson_rate'

def gpr(n_restarts=10):
    k=C(1.0,(1e-2,1e2))*RBF(1.0,(0.5,20.0))+WhiteKernel(0.1,(1e-2,1.0))
    return GaussianProcessRegressor(kernel=k,alpha=0.1,n_restarts_optimizer=n_restarts,normalize_y=True,random_state=SEED)

def bc_to_log(p,lam,eps=1e-10):
    if abs(lam)<0.01: return p
    return np.log(np.clip(np.power(np.clip(p*lam+1,eps,None),1.0/lam),eps,None))

def ks(X,n):
    d=cdist(X,X); i1,i2=np.unravel_index(np.argmax(d),d.shape); sel=[int(i1),int(i2)]
    rem=list(set(range(len(X)))-set(sel))
    while len(sel)<n:
        dd=cdist(X[rem],X[sel]); b=int(np.argmax(np.min(dd,axis=1))); sel.append(rem[b]); rem.remove(rem[b])
    return sel

def load(D):
    df=pd.read_excel(f'{D}/dataset.xlsx'); ad=pd.read_csv(f'{D}/applicability_domain.csv')
    feats_all=[c for c in df.columns if c not in (TFO,THX)]
    keep_rows=np.where(df[feats_all].notna().all(axis=1))[0]
    df=df.iloc[keep_rows].reset_index(drop=True); ad=ad.iloc[keep_rows].reset_index(drop=True)
    for f in LOGF:
        if (df[f]>0).all(): df[f]=np.log(df[f])
    df=df.drop(columns=DROP); feats=[f for f in feats_all if f in df.columns]
    m=np.ones(len(df),bool); m[PRE]=False
    df=df[m].reset_index(drop=True); ad=ad[m].reset_index(drop=True)
    X=df[feats].values.astype(float)
    ksi=ks(StandardScaler().fit_transform(X),9); kmask=np.zeros(len(X),bool); kmask[ksi]=True
    MI=[feats.index(f) for f in MF]; FI=[feats.index(f) for f in FF]
    add=lambda Z: np.hstack([Z]+[Z[:,a:a+1]*Z[:,b:b+1] for a in MI for b in FI])
    return dict(df=df,ad=ad,feats=feats,X=X,kmask=kmask,add=add)

def outliers_from_pool(P, n_out):
    if n_out==0: return []
    Xp=P['X'][~P['kmask']]; raw=P['df'][TFO].values[~P['kmask']]
    bc,lam=boxcox(raw); g=gpr(5); pr=np.zeros(len(raw))
    for tr,te in LeaveOneOut().split(Xp):
        sc=StandardScaler(); m=copy.deepcopy(g); m.fit(sc.fit_transform(P['add'](Xp[tr])),bc[tr])
        pr[te]=m.predict(sc.transform(P['add'](Xp[te]))).ravel()[0]
    r=np.log(raw)-bc_to_log(pr,lam); return np.argsort(np.abs(r/r.std()))[-n_out:].tolist()

def lin_ccc(y,yp):
    cov=np.cov(y,yp,ddof=0)[0,1]; return float(2*cov/(y.var()+yp.var()+(y.mean()-yp.mean())**2))

def metrics(y,yp):
    return dict(Q2=1-np.sum((y-yp)**2)/np.sum((y-y.mean())**2), CCC=lin_ccc(y,yp),
                r=pearsonr(y,yp)[0], rho=spearmanr(y,yp)[0], RMSE=float(np.sqrt(np.mean((y-yp)**2))),
                Bias=float(np.mean(yp-y)))

import re as _re
def study_key(r):
    s=str(r).strip(); s=_re.sub(r'\d+$','',s); m=_re.search(r'(19|20)\d\d',s); yr=m.group(0) if m else ''
    a=_re.split(r'[\s,(.]',s.strip())[0].lower().replace('el',''); return f'{a}_{yr}'


In [ ]:
%%writefile repro/an/fastgp.py
# Gaussian process with the same kernel, bounds and optimiser as the scikit-learn model, used for the permutation tests.
import numpy as np
from scipy.linalg import cholesky, cho_solve, solve_triangular
from scipy.optimize import minimize
from scipy.spatial.distance import pdist, squareform, cdist

ALPHA = 0.1
LOGB = np.log(np.array([[1e-2, 1e2], [0.5, 20.0], [1e-2, 1.0]]))
TH0 = np.log(np.array([1.0, 1.0, 0.1]))
_L2PI = np.log(2 * np.pi)

def restart_points(n, seed=42):
    rs = np.random.RandomState(seed)
    return [rs.uniform(LOGB[:, 0], LOGB[:, 1]) for _ in range(n)]
_RP = restart_points(10)

def _nlml(theta, D2, y, eye):
    c, l, s2 = np.exp(theta)
    E = np.exp(-0.5 * D2 / (l * l)); K = c * E
    K.flat[::K.shape[0] + 1] += s2 + ALPHA
    try:
        L = cholesky(K, lower=True, check_finite=False)
    except np.linalg.LinAlgError:
        return np.inf, np.zeros(3)
    a = cho_solve((L, True), y, check_finite=False)
    lml = -0.5 * (y @ a) - np.log(np.diag(L)).sum() - 0.5 * len(y) * _L2PI
    tmp = np.outer(a, a) - cho_solve((L, True), eye, check_finite=False)
    cE = c * E
    g = 0.5 * np.array([np.sum(tmp * cE), np.sum(tmp * cE * D2) / (l * l), s2 * np.trace(tmp)])
    return -lml, -g

def fit(Xs, y, n_restarts=1, D2=None):
    m = y.mean(); s = y.std(); s = 1.0 if s == 0 else s; yn = (y - m) / s
    if D2 is None: D2 = squareform(pdist(Xs, 'sqeuclidean'))
    eye = np.eye(len(y)); best = None
    for th0 in [TH0] + _RP[:n_restarts]:
        r = minimize(_nlml, th0, args=(D2, yn, eye), method='L-BFGS-B', jac=True, bounds=LOGB)
        if best is None or r.fun < best.fun: best = r
    th = best.x; c, l, s2 = np.exp(th)
    K = c * np.exp(-0.5 * D2 / (l * l)); K.flat[::K.shape[0] + 1] += s2 + ALPHA
    L = cholesky(K, lower=True, check_finite=False); a = cho_solve((L, True), yn, check_finite=False)
    return dict(th=th, a=a, L=L, Xs=Xs, m=m, s=s, lml=-best.fun)

def predict(g, Xq=None, d2q=None, return_std=False):
    c, l, s2 = np.exp(g['th'])
    if d2q is None: d2q = cdist(Xq, g['Xs'], 'sqeuclidean')
    Ks = c * np.exp(-0.5 * d2q / (l * l)); mu = Ks @ g['a'] * g['s'] + g['m']
    if not return_std: return mu
    V = solve_triangular(g['L'], Ks.T, lower=True, check_finite=False)
    var = np.clip((c + s2) - np.sum(V * V, axis=0), 0, None) * g['s'] ** 2
    return mu, np.sqrt(var)

def loo_folds(Xp, add):
    out = []; n = len(Xp)
    for i in range(n):
        tr = np.r_[0:i, i + 1:n]; A = add(Xp[tr]); mu = A.mean(0); sd = A.std(0); sd[sd == 0] = 1.0
        Z = (A - mu) / sd; z = (add(Xp[i:i + 1]) - mu) / sd
        out.append((tr, squareform(pdist(Z, 'sqeuclidean')), cdist(z, Z, 'sqeuclidean'), Z))
    return out

def loo_predict(folds, ybc, n_restarts=1, return_std=False):
    pr = np.zeros(len(ybc)); sd = np.zeros(len(ybc))
    for i, (tr, D2, d2q, Z) in enumerate(folds):
        g = fit(Z, ybc[tr], n_restarts, D2)
        if return_std:
            a, b = predict(g, d2q=d2q, return_std=True); pr[i] = a[0]; sd[i] = b[0]
        else:
            pr[i] = predict(g, d2q=d2q)[0]
    return (pr, sd) if return_std else pr


In [ ]:
%%writefile repro/an/a1_basic.py
# Training pool and final model: leave-one-out, external validation, Golbraikh-Tropsha criteria, calibration, leverage.
import sys, json, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '.'); from common import *
from scipy.stats import norm, pearsonr, spearmanr

def wilson(k, n, a=0.05):
    z = norm.ppf(1 - a / 2); ph = k / n; den = 1 + z * z / n; c = (ph + z * z / (2 * n)) / den
    h = z * ((ph * (1 - ph) / n + z * z / (4 * n * n)) ** 0.5) / den; return [max(0, c - h), min(1, c + h)]

P = load('data'); pool = np.where(~P['kmask'])[0]; ext = np.where(P['kmask'])[0]
ad = P['ad']; R = {}
R['n_model'] = len(P['X']); R['n_pool'] = len(pool); R['n_ext'] = len(ext)
R['ext_samples'] = [(str(ad['Sample Name'].iloc[i]), ad['Matrix'].iloc[i], ad['Filler'].iloc[i]) for i in ext]
print('external:', R['ext_samples'])
drop = outliers_from_pool(P, 9); keep = np.array([i for i in range(len(pool)) if i not in set(drop)])
R['excluded9'] = [(str(ad['Sample Name'].iloc[pool[i]]), ad['Matrix'].iloc[pool[i]], ad['Filler'].iloc[pool[i]], ad['Ref.'].iloc[pool[i]]) for i in sorted(drop)]
print('excluded 9:'); [print('  ', x) for x in R['excluded9']]

def fit_eval(tr_idx, T, tag):
    Xtr = P['X'][tr_idx]; Xte = P['X'][ext]
    raw = P['df'][T].values[tr_idx]; ly = np.log(raw); lyt = np.log(P['df'][T].values[ext]); bc, lam = boxcox(raw)
    pr = np.zeros(len(raw)); sdv = np.zeros(len(raw))
    for a, b in LeaveOneOut().split(Xtr):
        sc = StandardScaler(); m = gpr(10); m.fit(sc.fit_transform(P['add'](Xtr[a])), bc[a])
        mu, s = m.predict(sc.transform(P['add'](Xtr[b])), return_std=True); pr[b] = mu[0]; sdv[b] = s[0]
    plog = bc_to_log(pr, lam); loo = metrics(ly, plog)
    sc = StandardScaler().fit(P['add'](Xtr)); g = gpr(10); g.fit(sc.transform(P['add'](Xtr)), bc)
    ins = bc_to_log(g.predict(sc.transform(P['add'](Xtr))), lam); R2in = 1 - np.sum((ly - ins) ** 2) / np.sum((ly - ly.mean()) ** 2)
    pe_bc, pe_sd = g.predict(sc.transform(P['add'](Xte)), return_std=True); pe = bc_to_log(pe_bc, lam)
    ss = np.sum((lyt - pe) ** 2)
    e = dict(r2_ext=pearsonr(lyt, pe)[0] ** 2, Q2F1=1 - ss / np.sum((lyt - ly.mean()) ** 2), Q2F2=1 - ss / np.sum((lyt - lyt.mean()) ** 2),
             Q2F3=1 - (ss / len(lyt)) / (np.sum((ly - ly.mean()) ** 2) / len(ly)), CCC_ext=lin_ccc(lyt, pe), RMSEP=float(np.sqrt(ss / len(lyt))),
             rho_ext=spearmanr(lyt, pe)[0], sd_obs=float(lyt.std()), sd_pred=float(pe.std()))
    k = np.sum(pe * lyt) / np.sum(pe ** 2); kp = np.sum(pe * lyt) / np.sum(lyt ** 2)
    R0 = 1 - np.sum((lyt - k * pe) ** 2) / np.sum((lyt - lyt.mean()) ** 2)
    R0p = 1 - np.sum((pe - kp * lyt) ** 2) / np.sum((pe - pe.mean()) ** 2)
    r2 = e['r2_ext']
    gt = dict(k=float(k), kp=float(kp), R0=float(R0), R0p=float(R0p), c3a=float((r2 - R0) / r2), c3b=float((r2 - R0p) / r2), c5=float(abs(R0 - R0p)),
              C1=bool(loo['Q2'] > 0.5), C2=bool(r2 > 0.6), C3=bool(((r2 - R0) / r2 < 0.1) or ((r2 - R0p) / r2 < 0.1)),
              C4=bool((0.85 <= k <= 1.15) or (0.85 <= kp <= 1.15)), C5=bool(abs(R0 - R0p) < 0.3))
    kd = np.sum(pe * lyt) / np.sum(lyt ** 2); kpd = np.sum(pe * lyt) / np.sum(pe ** 2); st = np.sum((lyt - lyt.mean()) ** 2)
    r0d = 1 - np.sum((lyt - kd * pe) ** 2) / st; r0pd = 1 - np.sum((lyt - kpd * pe) ** 2) / st
    gt['deposited'] = dict(k=float(kd), kp=float(kpd), R0sq=float(r0d), R0sqp=float(r0pd), C3=bool(abs(r2 - r0d) / r2 < 0.1 or abs(r2 - r0pd) / r2 < 0.1), C5=bool(abs(r0d - r0pd) < 0.3))
    def cal(mu, sd):
        noms = np.arange(0.05, 1.0, 0.05); emp = np.array([np.mean(np.abs(bc - mu) <= norm.ppf(0.5 + q / 2) * sd) for q in noms])
        out = dict(L1_19=float(np.mean(np.abs(emp - noms))), SQ_19=float(np.sum((emp - noms) ** 2)), emp=[float(x) for x in emp])
        for lv in (0.50, 0.68, 0.90, 0.95):
            kk = int(np.sum(np.abs(bc - mu) <= norm.ppf(0.5 + lv / 2) * sd)); out[f'cov{int(lv*100)}'] = dict(k=kk, n=len(bc), frac=kk / len(bc), wilson=wilson(kk, len(bc)))
        pobs = norm.cdf((bc - mu) / sd)
        for nb in (10, 19):
            q = np.linspace(1 / nb, 1, nb); em = np.array([(pobs <= x).mean() for x in q]); out[f'L1_cdf{nb}'] = float(np.mean(np.abs(q - em))); out[f'SQ_cdf{nb}'] = float(np.sum((q - em) ** 2))
        out['mean_sd'] = float(sd.mean()); return out
    calI = cal(pr, sdv)
    Z = sc.transform(P['add'](Xtr)); mu2 = np.zeros(len(raw)); sd2 = np.zeros(len(raw))
    for a, b in LeaveOneOut().split(Z):
        gg = GaussianProcessRegressor(kernel=g.kernel_, alpha=g.alpha, normalize_y=True, optimizer=None); gg.fit(Z[a], bc[a])
        mm, s2 = gg.predict(Z[b], return_std=True); mu2[b] = mm[0]; sd2[b] = s2[0]
    calII = cal(mu2, sd2); q2_fix = 1 - np.sum((bc - mu2) ** 2) / np.sum((bc - bc.mean()) ** 2)
    s8 = StandardScaler().fit(Xtr); Dt = np.column_stack([np.ones(len(Xtr)), s8.transform(Xtr)]); De = np.column_stack([np.ones(len(Xte)), s8.transform(Xte)])
    G = np.linalg.pinv(Dt.T @ Dt); ht = np.einsum('ij,jk,ik->i', Dt, G, Dt); he = np.einsum('ij,jk,ik->i', De, G, De); hs = 3 * Dt.shape[1] / len(Xtr)
    rs = bc - mu2; srs = rs / rs.std()
    A = dict(h_star=float(hs), train_lev_in=int(np.sum(ht <= hs)), train_in_AD=int(np.sum((ht <= hs) & (np.abs(srs) <= 3))), ext_lev_in=int(np.sum(he <= hs)),
             ext_h=[float(x) for x in he], h_star_23=float(3 * 24 / len(Xtr)))
    out = dict(n=len(raw), lam=float(lam), kernel=str(g.kernel_), R2_in=float(R2in), loo={k2: float(v) for k2, v in loo.items()}, ext={k2: float(v) for k2, v in e.items()},
               GT=gt, calI=calI, calII=calII, Q2_fixed_kernel_bc=float(q2_fix), AD=A, loo_pred=[float(x) for x in plog], y=[float(x) for x in ly], ext_pred=[float(x) for x in pe], ext_y=[float(x) for x in lyt], ext_sd_bc=[float(x) for x in pe_sd])
    print(f"{tag} {T[:5]} n={len(raw)} lam={lam:.4f} {g.kernel_}\n   R2in={R2in:.3f} LOO Q2={loo['Q2']:.3f} CCC={loo['CCC']:.3f} r={loo['r']:.3f} rho={loo['rho']:.3f} RMSE={loo['RMSE']:.3f} bias={loo['Bias']:.3f}"
          f"\n   ext r2={e['r2_ext']:.3f} F1={e['Q2F1']:.3f} F2={e['Q2F2']:.3f} F3={e['Q2F3']:.3f} CCCext={e['CCC_ext']:.3f} RMSEP={e['RMSEP']:.3f} rho={e['rho_ext']:.3f} sd_obs={e['sd_obs']:.2f} sd_pred={e['sd_pred']:.2f}"
          f"\n   GT std: k={k:.3f} k'={kp:.3f} R0={R0:.3f} R0'={R0p:.3f} c3={min(gt['c3a'], gt['c3b']):.3f} c5={gt['c5']:.2f} -> C1..C5 {[gt[c] for c in ('C1','C2','C3','C4','C5')]} | deposited impl: {gt['deposited']}"
          f"\n   cal I (per-fold refit): cov95 {calI['cov95']['k']}/{len(raw)} L1_19={calI['L1_19']:.3f} SQ19={calI['SQ_19']:.3f} | cal II (fixed kernel): cov50/68/90/95 {[calII[c]['k'] for c in ('cov50','cov68','cov90','cov95')]} L1_19={calII['L1_19']:.3f} L1cdf10={calII['L1_cdf10']:.3f} SQcdf10={calII['SQ_cdf10']:.3f} L1cdf19={calII['L1_cdf19']:.3f} SQcdf19={calII['SQ_cdf19']:.3f}"
          f"\n   AD: h*={hs:.3f} train lev in {A['train_lev_in']} in AD {A['train_in_AD']} ext in {A['ext_lev_in']}/9", flush=True)
    return out

for tag, idx in (('pool49', pool), ('final40', pool[keep])):
    R[tag] = {T: fit_eval(idx, T, tag) for T in (TFO, THX)}
json.dump(R, open('out/a1_basic.json', 'w'), indent=1, default=str)


In [ ]:
%%writefile repro/an/a2_accounting.py
# Sample accounting and Mahalanobis distances.
import sys, json, re, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '.'); from common import *
from scipy.stats import chi2, pearsonr, spearmanr
from sklearn.impute import SimpleImputer
d = pd.read_excel('data/dataset.xlsx'); a = pd.read_csv('data/applicability_domain.csv'); R = {}
feats21 = [c for c in d.columns if c not in (TFO, THX)]
print('rows', len(d), 'descriptors', len(feats21), 'rows with a missing descriptor:', list(d.index[d[feats21].isna().any(axis=1)]), [c for c in feats21 if d[c].isna().any()])
def maha(M):
    Z = StandardScaler().fit_transform(SimpleImputer(strategy='mean').fit_transform(M)); mu = Z.mean(0); S = np.linalg.inv(np.cov(Z.T))
    return np.sqrt(np.einsum('ij,jk,ik->i', Z - mu, S, Z - mu))
for nm, cols in (('23 numeric columns (deposited notebook)', list(d.columns)), ('21 descriptors', feats21)):
    D = maha(d[cols].values); o = np.argsort(-D); thr = np.percentile(D, 95)
    print(f"\nMahalanobis on {nm}: 95th percentile = {thr:.3f}; max attainable (n-1)/sqrt(n) = {(len(d)-1)/np.sqrt(len(d)):.3f}")
    print('  top 8:', [(int(i), round(float(D[i]), 3)) for i in o[:8]]); print('  flagged (> 95th pct):', sorted(int(i) for i in np.where(D > thr)[0]))
    p = len(cols); print(f"  chi2({p},0.99)={chi2.ppf(0.99,p):.1f} -> D>{np.sqrt(chi2.ppf(0.99,p)):.2f}: {sorted(int(i) for i in np.where(D**2 > chi2.ppf(0.99,p))[0])};  chi2({p},0.95)={chi2.ppf(0.95,p):.1f} -> {sorted(int(i) for i in np.where(D**2 > chi2.ppf(0.95,p))[0])}")
    R[nm] = dict(thr95=float(thr), top=[(int(i), float(D[i])) for i in o[:8]])
P = load('data')
dd = d.dropna(subset=feats21).reset_index(drop=True)
for f in LOGF: dd[f] = np.log(dd[f])
D8 = maha(dd[P['feats']].values)
print('\n8 retained (log-transformed) descriptors, 61 rows: rows with D^2 > 20.09:', [(int(i), round(float(D8[i]**2), 1)) for i in np.where(D8**2 > chi2.ppf(0.99, 8))[0]], '| top 6 D^2:', [(int(i), round(float(D8[i]**2), 1)) for i in np.argsort(-D8)[:6]])
Z = (d[feats21] - d[feats21].mean()) / d[feats21].std()
for r in (0, 8, 18, 25):
    zz = Z.loc[r].abs().sort_values(ascending=False).head(3); print(f"row {r} ({a['Matrix'][r]} / {a['Filler'][r]}): largest |z|:", [(k, round(float(Z.loc[r, k]), 1)) for k in zz.index])
def sk(r):
    s = str(r).strip(); s = re.sub(r'\[?\d+\]?$', '', s).strip(); m = re.search(r'(19|20)\d\d', s); yr = m.group(0) if m else ''
    au = re.split(r'[\s,(.]', s.strip())[0].lower(); return f'{au}_{yr}'
a['study'] = a['Ref.'].map(sk)
print('\nstudies in the 62:', a['study'].nunique(), '| composites per study:', a['study'].value_counts().value_counts().sort_index().to_dict())
print(sorted(a['study'].unique()))
ad = P['ad'].copy(); ad['study'] = ad['Ref.'].map(sk); pool = np.where(~P['kmask'])[0]
print('studies in the 58:', ad['study'].nunique(), '| in pool 49:', ad['study'].iloc[pool].nunique(), '| matrices in 62:', a['Matrix'].nunique(), 'fillers:', a['Filler'].nunique())
drop = outliers_from_pool(P, 9); keep = np.array([i for i in range(len(pool)) if i not in set(drop)]); fin = pool[keep]
print('studies in final 40:', ad['study'].iloc[fin].nunique())
mat = ad['Matrix'].replace({'PBSu': 'PBS', 'Bio-PBS': 'PBS'}); cnt = mat.iloc[pool].value_counts()
print('\nmatrix counts in pool (PBS names merged):', cnt.to_dict())
ex = pool[sorted(drop)]
print('excluded composites and the number of pool composites sharing their matrix:', [(str(ad['Sample Name'].iloc[i]), mat.iloc[i], int(cnt[mat.iloc[i]])) for i in ex])
rare = np.array([cnt[mat.iloc[i]] <= 2 for i in pool]); isex = np.array([i in set(ex) for i in pool])
print(f"pool composites with matrix represented <=2 times: {rare.sum()}, of which excluded {int((rare & isex).sum())}; others {int((~rare).sum())}, of which excluded {int((~rare & isex).sum())}")
for nm, idx in (('58', np.arange(len(P['X']))), ('pool49', pool), ('final40', fin)):
    X = pd.DataFrame(P['X'][idx], columns=P['feats']); c = X.corr()
    print(f"\n[{nm}] r(logP_matrix, ln matrix_CED) = {c.loc['logP_matrix','matrix_CED']:+.3f}; r(logP_matrix, ln interface_rot) = {c.loc['logP_matrix','interface_rot']:+.3f}; r(logP, ln half-life) = {c.loc['logP_matrix','Half life_matrix_avg']:+.3f}; r(ln tort, ln dia) = {c.loc['tortuosity','fiber_dia_or_average_particle_size_mm']:+.3f}; max |r| off-diag = {np.abs(c.values - np.eye(8)).max():.3f}")
raw = d.dropna(subset=feats21).reset_index(drop=True)
print('untransformed, 61 rows: r(logP_matrix, matrix_CED) = %.3f ; r(logP_matrix, interface_rot) = %.3f' % (raw['logP_matrix'].corr(raw['matrix_CED']), raw['logP_matrix'].corr(raw['interface_rot'])))
print('\nrate ranges (62): FO %.2e..%.2e (%.2f orders), HC %.2e..%.2e (%.2f orders)' % (d[TFO].min(), d[TFO].max(), np.log10(d[TFO].max()/d[TFO].min()), d[THX].min(), d[THX].max(), np.log10(d[THX].max()/d[THX].min())))
json.dump(R, open('out/a2_accounting.json', 'w'), indent=1)


In [ ]:
%%writefile repro/an/a7_misc.py
# Baselines, residuals of the training pool, external interval coverage, direction of effects.
import sys, json, itertools, math, numpy as np, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '.'); from common import *; import fastgp as fg
from scipy.stats import norm, spearmanr
P = load('data'); pool = np.where(~P['kmask'])[0]; ext = np.where(P['kmask'])[0]; ad = P['ad']; F = P['feats']
drop = outliers_from_pool(P, 9); keep = np.array([i for i in range(len(pool)) if i not in set(drop)]); fin = pool[keep]
def canon(m): return 'PBS' if m in ('PBSu', 'Bio-PBS', 'PBS') else m
mats = np.array([canon(m) for m in ad['Matrix'].values]); out = {}
def loo(idx, T, nr=10, std=False):
    X = P['X'][idx]; raw = P['df'][T].values[idx]; bc, lam = boxcox(raw); folds = fg.loo_folds(X, P['add'])
    r = fg.loo_predict(folds, bc, n_restarts=nr, return_std=std); pr = r[0] if std else r
    return np.log(raw), bc_to_log(pr, lam), bc, (r if std else None), lam
yl, pl, *_ = loo(pool, TFO); res = yl - pl; z = res / res.std(); order = np.argsort(-np.abs(z))
cnt = {m: int((mats[pool] == m).sum()) for m in set(mats[pool])}
out['pool_resid'] = [dict(rank=int(k + 1), sample=str(ad['Sample Name'].iloc[pool[i]]), matrix=mats[pool[i]], filler=ad['Filler'].iloc[pool[i]], z=float(z[i]), n_matrix=cnt[mats[pool[i]]],
                          excluded=bool(i in set(drop))) for k, i in enumerate(order[:14])]
print('largest |standardised LOO residual| in the pool (first-order):'); [print('  ', d) for d in out['pool_resid']]
print('  n with |z|>2:', int((np.abs(z) > 2).sum()), ' |z|>1.5:', int((np.abs(z) > 1.5).sum()), ' |z|>1:', int((np.abs(z) > 1).sum()), ' 9th largest |z|: %.2f, 10th: %.2f' % (np.sort(np.abs(z))[-9], np.sort(np.abs(z))[-10]))
for thr in (2, 3):
    ok = np.array([cnt[mats[i]] >= thr for i in pool]); idx = pool[ok]; o = {}
    for T in (TFO, THX):
        yl, pl, *_ = loo(idx, T); o[T] = {k: float(v) for k, v in metrics(yl, pl).items()}
    out[f'matrix_ge{thr}'] = dict(n=int(ok.sum()), n_excl_overlap=int(sum((not ok[i]) and (i in set(drop)) for i in range(len(pool)))), **o)
    print(f'matrix represented >= {thr} times: n = {ok.sum()}', {T[:5]: (round(o[T]["Q2"], 3), round(o[T]["CCC"], 3)) for T in o})
for T in (TFO, THX):
    yl, pl, bc, _, lam = loo(fin, T); lp = P['df']['logP_matrix'].values[fin]; n = len(fin); m40 = mats[fin]
    look = np.zeros(n); quin = np.zeros(n)
    for i in range(n):
        tr = np.r_[0:i, i + 1:n]; same = tr[m40[tr] == m40[i]]; look[i] = yl[same].mean() if len(same) else yl[tr].mean()
    qe = np.quantile(lp, [0.2, 0.4, 0.6, 0.8]); qb = np.digitize(lp, qe)
    for i in range(n):
        tr = np.r_[0:i, i + 1:n]; same = tr[qb[tr] == qb[i]]; quin[i] = yl[same].mean() if len(same) else yl[tr].mean()
    q2 = lambda p: float(1 - np.sum((yl - p) ** 2) / np.sum((yl - yl.mean()) ** 2))
    out['base_' + T] = dict(q2_gpr=q2(pl), q2_lookup=q2(look), ccc_lookup=lin_ccc(yl, look), q2_quintile=q2(quin), gpr_better_than_lookup=int(np.sum(np.abs(yl - pl) < np.abs(yl - look))),
                            gpr_better_than_quintile=int(np.sum(np.abs(yl - pl) < np.abs(yl - quin))), n=n, n_singletons=int(sum((m40 == m).sum() == 1 for m in m40)))
    print(T[:5], out['base_' + T])
for T in (TFO, THX):
    X = P['X'][fin]; raw = P['df'][T].values[fin]; bc, lam = boxcox(raw); A = P['add'](X); mu = A.mean(0); sd = A.std(0); Z = (A - mu) / sd
    g = fg.fit(Z, bc, 10); Ze = (P['add'](P['X'][ext]) - mu) / sd; m, s = fg.predict(g, Ze, return_std=True)
    ye = P['df'][T].values[ext]; ybc = (ye ** lam - 1) / lam; zsc = (ybc - m) / s
    out['extcov_' + T] = dict(inside95=int(np.sum(np.abs(zsc) <= norm.ppf(0.975))), inside90=int(np.sum(np.abs(zsc) <= norm.ppf(0.95))), z=[float(v) for v in zsc], kernel=[float(v) for v in np.exp(g['th'])])
    print(T[:5], 'external inside 95% interval:', out['extcov_' + T]['inside95'], '/ 9; z =', np.round(zsc, 2), 'kernel', np.round(np.exp(g['th']), 4))
def shapley(f, X):
    m = X.shape[1]; n = len(X); v = {}
    for r in range(m + 1):
        for S in itertools.combinations(range(m), r):
            Zb = np.repeat(X[None, :, :], n, axis=0)
            if S: Zb[:, :, list(S)] = X[:, None, list(S)]
            v[S] = f(Zb.reshape(n * n, -1)).reshape(n, n).mean(1)
    phi = np.zeros((n, m))
    for j in range(m):
        oth = [k for k in range(m) if k != j]
        for r in range(m):
            w = math.factorial(r) * math.factorial(m - r - 1) / math.factorial(m)
            for S in itertools.combinations(oth, r): phi[:, j] += w * (v[tuple(sorted(S + (j,)))] - v[S])
    return phi
for T in (TFO, THX):
    X = P['X'][fin]; raw = P['df'][T].values[fin]; bc, lam = boxcox(raw); A = P['add'](X); mu = A.mean(0); sd = A.std(0); g = fg.fit((A - mu) / sd, bc, 10)
    f = lambda Zq: bc_to_log(fg.predict(g, (P['add'](Zq) - mu) / sd), lam); ph = shapley(f, X)
    out['dir_' + T] = {F[j]: dict(mean_abs=float(np.abs(ph[:, j]).mean()), rho_value_vs_shap=float(spearmanr(X[:, j], ph[:, j])[0])) for j in range(len(F))}
    print(T[:5], {F[j][:14]: (round(out['dir_' + T][F[j]]['mean_abs'], 3), round(out['dir_' + T][F[j]]['rho_value_vs_shap'], 2)) for j in range(len(F))})
json.dump(out, open('out/a7_misc.json', 'w'), indent=1)


In [ ]:
%%writefile repro/an/a9_prelim.py
# Sensitivity to the number of structurally isolated composites removed before modelling.
import sys, json, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '.'); import common; from common import *; import fastgp as fg
from sklearn.impute import SimpleImputer
d = pd.read_excel('data/dataset.xlsx'); feats21 = [c for c in d.columns if c not in (TFO, THX)]
Z = StandardScaler().fit_transform(SimpleImputer(strategy='mean').fit_transform(d[feats21].values)); S = np.linalg.inv(np.cov(Z.T)); D = np.sqrt(np.einsum('ij,jk,ik->i', Z - Z.mean(0), S, Z - Z.mean(0)))
miss = list(d.index[d[feats21].isna().any(axis=1)]); order = [int(i) for i in np.argsort(-D) if i not in miss]
print('incomplete rows:', miss, '| most isolated (row, D):', [(i, round(float(D[i]), 2)) for i in order[:7]], '| 95th percentile', round(float(np.percentile(D, 95)), 3))
def reindex(rows): return [r - sum(m < r for m in miss) for r in rows]
out = {}
for k in range(0, 6):
    common.PRE = reindex(order[:k]); P = load('data'); pool = np.where(~P['kmask'])[0]; ext = np.where(P['kmask'])[0]
    drop = outliers_from_pool(P, 9); fin = pool[[i for i in range(len(pool)) if i not in set(drop)]]; o = dict(removed=order[:k], next_D=float(D[order[k]]), n_model=len(P['X']), n_pool=len(pool), n_final=len(fin))
    for T in (TFO, THX):
        X = P['X'][fin]; raw = P['df'][T].values[fin]; bc, lam = boxcox(raw); pr = fg.loo_predict(fg.loo_folds(X, P['add']), bc, 10); m = metrics(np.log(raw), bc_to_log(pr, lam))
        Xp = P['X'][pool]; rawp = P['df'][T].values[pool]; bcp, lamp = boxcox(rawp); prp = fg.loo_predict(fg.loo_folds(Xp, P['add']), bcp, 10); mp = metrics(np.log(rawp), bc_to_log(prp, lamp))
        o[T] = dict(Q2=float(m['Q2']), CCC=float(m['CCC']), Q2_pool=float(mp['Q2']), CCC_pool=float(mp['CCC']))
    out[k] = o; print(k, o, flush=True)
json.dump(out, open('out/a9_prelim.json', 'w'), indent=1)


In [ ]:
%%writefile repro/an/a6_shap.py
# Bootstrap of the Shapley ranking with exact Shapley values.
import sys, json, itertools, math, time, numpy as np, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '.'); from common import *
MODEL, NBOOT, OUT = sys.argv[1], int(sys.argv[2]), sys.argv[3]
P = load('data'); pool = np.where(~P['kmask'])[0]; F = P['feats']
drop = outliers_from_pool(P, 9 if MODEL == 'final40' else 0); keep = np.array([i for i in range(len(pool)) if i not in set(drop)]); idx = pool[keep]
X = P['X'][idx]; n = len(X); ix = {f: F.index(f) for f in F}
SHORT = {'Half life_matrix_avg': 'matrix half-life', 'Hemicellulose_Percentage': 'hemicellulose', 'tortuosity': 'tortuosity', 'logP_matrix': 'logP_matrix',
         'interface_rot': 'interface_rot', 'fiber_sol_parameter': 'fibre sol. parameter', 'matrix_CED': 'matrix CED', 'fiber_dia_or_average_particle_size_mm': 'fibre diameter'}
def players(scheme):
    single = lambda f: (SHORT[f], [ix[f]])
    if scheme == 'single': return [single(f) for f in F]
    if scheme == 'H2':
        return [('matrix hydrophobicity (logP + CED)', [ix['logP_matrix'], ix['matrix_CED']])] + [single(f) for f in F if f not in ('logP_matrix', 'matrix_CED')]
    if scheme == 'M3':
        return [('matrix chemistry (logP + CED + half-life)', [ix[f] for f in MF])] + [single(f) for f in F if f not in MF]
    if scheme == 'G4':
        return [('matrix chemistry', [ix[f] for f in MF]), ('fibre geometry (tortuosity + diameter)', [ix['tortuosity'], ix['fiber_dia_or_average_particle_size_mm']]),
                ('fibre chemistry (hemicellulose + sol. parameter)', [ix['Hemicellulose_Percentage'], ix['fiber_sol_parameter']]), ('interface_rot', [ix['interface_rot']])]
def shapley(f, Xe, Xb, pl):
    m = len(pl); ne, nb = len(Xe), len(Xb); v = {}
    for r in range(m + 1):
        for S in itertools.combinations(range(m), r):
            cols = [c for j in S for c in pl[j][1]]
            Z = np.repeat(Xb[None, :, :], ne, axis=0)
            if cols: Z[:, :, cols] = Xe[:, None, cols]
            v[S] = f(Z.reshape(ne * nb, -1)).reshape(ne, nb).mean(1)
    phi = np.zeros((ne, m))
    for j in range(m):
        others = [k for k in range(m) if k != j]
        for r in range(m):
            w = math.factorial(r) * math.factorial(m - r - 1) / math.factorial(m)
            for S in itertools.combinations(others, r):
                phi[:, j] += w * (v[tuple(sorted(S + (j,)))] - v[S])
    return phi
def fit(Xtr, ybc, restarts=10):
    sc = StandardScaler().fit(P['add'](Xtr)); g = gpr(restarts); g.fit(sc.transform(P['add'](Xtr)), ybc)
    return lambda Z: g.predict(sc.transform(P['add'](Z)))
SCHEMES = ('single', 'H2', 'M3', 'G4'); res = {}
for T in (TFO, THX):
    raw = P['df'][T].values[idx]; bc, lam = boxcox(raw); res[T] = {}
    f = fit(X, bc)
    for sch in SCHEMES:
        pl = players(sch); ph = shapley(f, X, X, pl); ma = np.abs(ph).mean(0)
        phl = shapley(lambda Z: bc_to_log(f(Z), lam), X, X, pl); mal = np.abs(phl).mean(0)
        res[T][sch] = dict(players=[p[0] for p in pl], mean_abs_bc=[float(x) for x in ma], mean_abs_log=[float(x) for x in mal],
                           order_log=[pl[i][0] for i in np.argsort(-mal)], share_log=[float(x) for x in mal / mal.sum()])
    print(f"[{MODEL} {T[:5]}] full sample, log scale:", [(a, round(b, 3)) for a, b in zip(res[T]['single']['players'], res[T]['single']['mean_abs_log'])], flush=True)
    print("   order:", res[T]['single']['order_log'], '| H2 order:', res[T]['H2']['order_log'][:3], [round(x, 3) for x in sorted(res[T]['H2']['mean_abs_log'], reverse=True)[:3]], flush=True)
    rng = np.random.default_rng(SEED); cnt = {sch: dict(top1=np.zeros(len(players(sch))), top2=np.zeros(len(players(sch))), top3=np.zeros(len(players(sch)))) for sch in SCHEMES}
    hist = {sch: [] for sch in SCHEMES}; t0 = time.time()
    for it in range(NBOOT):
        bi = rng.integers(0, n, size=n); Xb = X[bi]; yb = bc[bi]; fb = fit(Xb, yb)
        for sch in SCHEMES:
            pl = players(sch); ma = np.abs(shapley(fb, Xb, Xb, pl)).mean(0); o = np.argsort(-ma); hist[sch].append([float(x) for x in ma])
            cnt[sch]['top1'][o[0]] += 1; cnt[sch]['top2'][o[:2]] += 1; cnt[sch]['top3'][o[:3]] += 1
        if (it + 1) in (60, 200, 500, 1000, NBOOT):
            for sch in SCHEMES:
                res[T][sch][f'boot{it + 1}'] = {k: [float(x) / (it + 1) * 100 for x in v] for k, v in cnt[sch].items()}
            s = res[T]['single'][f'boot{it + 1}']; h = res[T]['H2'][f'boot{it + 1}']; m3 = res[T]['M3'][f'boot{it + 1}']; g4 = res[T]['G4'][f'boot{it + 1}']
            nm = res[T]['single']['players']
            print(f"   B={it + 1} ({time.time() - t0:.0f}s) single rank-1 %: " + ', '.join(f"{a} {b:.1f}" for a, b in zip(nm, s['top1']) if b > 0) +
                  "\n        single top-3 %: " + ', '.join(f"{a} {b:.1f}" for a, b in zip(nm, s['top3'])) +
                  f"\n        H2 rank-1 %: {h['top1'][0]:.1f} (tortuosity {h['top1'][res[T]['H2']['players'].index('tortuosity')]:.1f}) | M3 rank-1 %: {m3['top1'][0]:.1f} | G4 rank-1 %: " +
                  ', '.join(f"{a.split(' (')[0]} {b:.1f}" for a, b in zip(res[T]['G4']['players'], g4['top1'])), flush=True)
    res[T]['boot_hist'] = {sch: hist[sch] for sch in ('single', 'H2')}
json.dump(res, open(OUT, 'w'), indent=0)


In [ ]:
%%writefile repro/an/a4f_yrand_fast.py
# y-randomisation with the complete pipeline refitted for every permutation.
import sys, json, os, time, numpy as np, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '.'); from common import *; import fastgp as fg
START, END, OUT = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3]
P = load('data'); pool = np.where(~P['kmask'])[0]
drop = outliers_from_pool(P, 9); keep = np.array([i for i in range(len(pool)) if i not in set(drop)])
Xp = P['X'][pool][keep]; folds = fg.loo_folds(Xp, P['add']); TG = (TFO, THX)
RAW = {T: P['df'][T].values[pool][keep] for T in TG}; BC = {T: boxcox(RAW[T]) for T in TG}; YL = {T: np.log(RAW[T]) for T in TG}
def stat(T, perm, nr=1):
    bc, lam = BC[T]; pr = fg.loo_predict(folds, bc[perm], n_restarts=nr); yl = YL[T][perm]; yp = bc_to_log(pr, lam)
    return 1 - np.sum((yl - yp) ** 2) / np.sum((yl - yl.mean()) ** 2), lin_ccc(yl, yp)
ck = OUT + '.npz'
if os.path.exists(ck):
    z = np.load(ck); seeds = list(z['seeds']); q = {T: list(z['q2_' + T]) for T in TG}; c = {T: list(z['ccc_' + T]) for T in TG}
else:
    seeds = []; q = {T: [] for T in TG}; c = {T: [] for T in TG}
def save(): np.savez(ck, seeds=np.array(seeds), **{'q2_' + T: np.array(q[T]) for T in TG}, **{'ccc_' + T: np.array(c[T]) for T in TG})
done = set(seeds); t0 = time.time(); n_new = 0
for s in range(START, END + 1):
    if s in done: continue
    perm = np.random.default_rng(s).permutation(len(Xp))
    for T in TG:
        a, b = stat(T, perm); q[T].append(a); c[T].append(b)
    seeds.append(s); n_new += 1
    if n_new % 100 == 0:
        save(); print(f"{len(seeds)} done ({(time.time() - t0) / n_new:.2f} s/perm) max Q2: " + ', '.join(f"{T[:5]} {max(q[T]):.3f}" for T in TG), flush=True)
save(); ident = np.arange(len(Xp)); obs = {T: stat(T, ident, nr=10) for T in TG}
json.dump({T: dict(n=len(Xp), nperm=len(seeds), obs_q2=float(obs[T][0]), obs_ccc=float(obs[T][1]), null_mean=float(np.mean(q[T])), null_sd=float(np.std(q[T])),
                   null_min=float(np.min(q[T])), null_max=float(np.max(q[T])), null_ccc_max=float(np.max(c[T])), b=int(np.sum(np.array(q[T]) >= obs[T][0]))) for T in TG},
          open(OUT + '.json', 'w'), indent=1)
print('finished', OUT, len(seeds), flush=True)


In [ ]:
%%writefile repro/an/a8_selperm.py
# Permutation test in which the response-outlier screening is repeated for every permutation.
import sys, json, os, time, numpy as np, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '.'); from common import *; import fastgp as fg
START, END, OUT = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3]
P = load('data'); pool = np.where(~P['kmask'])[0]; Xp = P['X'][pool]; n = len(pool); TG = (TFO, THX)
RAW = {T: P['df'][T].values[pool] for T in TG}; FOLDS49 = fg.loo_folds(Xp, P['add'])
def procedure(perm, nr=1, nr_out=1):
    raw = RAW[TFO][perm]; bc, lam = boxcox(raw); pr = fg.loo_predict(FOLDS49, bc, n_restarts=nr_out); r = np.log(raw) - bc_to_log(pr, lam)
    drop = set(np.argsort(np.abs(r / r.std()))[-9:].tolist()); keep = np.array([i for i in range(n) if i not in drop]); folds = fg.loo_folds(Xp[keep], P['add']); out = {}
    for T in TG:
        rw = RAW[T][perm][keep]; b, l = boxcox(rw); yl = np.log(rw); yp = bc_to_log(fg.loo_predict(folds, b, n_restarts=nr), l)
        out[T] = (1 - np.sum((yl - yp) ** 2) / np.sum((yl - yl.mean()) ** 2), lin_ccc(yl, yp))
    return out
ck = OUT + '.npz'
if os.path.exists(ck):
    z = np.load(ck); seeds = list(z['seeds']); q = {T: list(z['q2_' + T]) for T in TG}; c = {T: list(z['ccc_' + T]) for T in TG}
else:
    seeds = []; q = {T: [] for T in TG}; c = {T: [] for T in TG}
def save(): np.savez(ck, seeds=np.array(seeds), **{'q2_' + T: np.array(q[T]) for T in TG}, **{'ccc_' + T: np.array(c[T]) for T in TG})
done = set(seeds); t0 = time.time(); k = 0
for s in range(START, END + 1):
    if s in done: continue
    o = procedure(np.random.default_rng(100000 + s).permutation(n))
    for T in TG: q[T].append(o[T][0]); c[T].append(o[T][1])
    seeds.append(s); k += 1
    if k % 50 == 0:
        save(); print(f"{len(seeds)} done ({(time.time() - t0) / k:.2f} s/perm) null Q2 mean/max: " + ', '.join(f"{T[:5]} {np.mean(q[T]):.3f}/{max(q[T]):.3f}" for T in TG), flush=True)
save()
if START == 1:
    obs = procedure(np.arange(n), nr=10, nr_out=5); json.dump({T: dict(q2=float(obs[T][0]), ccc=float(obs[T][1])) for T in TG}, open(OUT + '_obs.json', 'w')); print('observed', obs, flush=True)
print('finished', OUT, len(seeds), flush=True)


In [ ]:
%%writefile repro/an/a8_merge.py
# Summary of the screening-aware permutation test.
import sys, glob, json, numpy as np
A1 = json.load(open('out/a1_basic.json'))['final40']; obs = {'first_order_rate': A1['first_order_rate']['loo']['Q2'], 'hixson_rate': A1['hixson_rate']['loo']['Q2']}
zs = [np.load(f) for f in sorted(glob.glob('out/a8_w*.npz'))]; seeds = np.concatenate([z['seeds'] for z in zs]); assert len(set(seeds.tolist())) == len(seeds)
mx = int(sys.argv[1]) if len(sys.argv) > 1 else int(seeds.max()); keep = seeds <= mx; n = int(keep.sum()); assert sorted(seeds[keep].tolist()) == list(range(1, n + 1)), 'seeds are not a complete range 1..n'
out = dict(n=n)
for T, t in (('first_order_rate', 'fo'), ('hixson_rate', 'hc')):
    q = np.concatenate([z['q2_' + T] for z in zs])[keep]; c = np.concatenate([z['ccc_' + T] for z in zs])[keep]
    out.update({f'b_{t}': int(np.sum(q >= obs[T])), f'mean_{t}': float(q.mean()), f'sd_{t}': float(q.std()), f'p95_{t}': float(np.percentile(q, 95)), f'p99_{t}': float(np.percentile(q, 99)), f'max_{t}': float(q.max()),
                f'obs_{t}': float(obs[T]), f'ccc_max_{t}': float(c.max()), f'ccc_b_{t}': int(np.sum(c >= A1[T]['loo']['CCC']))})
try:
    o = json.load(open('out/a8_w1_obs.json')); out['obs_check'] = {T: o[T]['q2'] for T in o}
    assert all(abs(o[T]['q2'] - obs[T]) < 1e-6 for T in o), ('observed statistic of the procedure differs from the final model', o, obs)
except FileNotFoundError: out['obs_check'] = None
json.dump(out, open('out/a8_selperm.json', 'w'), indent=1); print(out)


In [ ]:
%%writefile repro/an/a10_extra.py
# Fit times of the model families and effect of the number of optimiser restarts.
import sys, json, time, numpy as np, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '.'); from common import *; import fastgp as fg
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.gaussian_process.kernels import Matern
P = load('data'); pool = np.where(~P['kmask'])[0]; out = {}
def gp_k(kind):
    base = {'rbf': RBF(1.0, (0.5, 20.0)), 'm15': Matern(1.0, (0.5, 20.0), nu=1.5), 'm25': Matern(1.0, (0.5, 20.0), nu=2.5)}[kind]
    return GaussianProcessRegressor(kernel=C(1.0, (1e-2, 1e2)) * base + WhiteKernel(0.1, (1e-2, 1.0)), alpha=0.1, n_restarts_optimizer=0, normalize_y=True, random_state=SEED)
class GPKernelSelect:
    def fit(self, X, y):
        best = None
        for kd in ('rbf', 'm15', 'm25'):
            g = gp_k(kd).fit(X, y); lml = g.log_marginal_likelihood_value_
            if best is None or lml > best[0]: best = (lml, g)
        self.m = best[1]; return self
inner = KFold(5, shuffle=True, random_state=SEED); mse = 'neg_mean_squared_error'
FAM = {
 'Ridge': (lambda: Ridge(), lambda: GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1, 10, 100, 1000]}, cv=inner, scoring=mse)),
 'Lasso': (lambda: Lasso(), lambda: GridSearchCV(Lasso(max_iter=20000), {'alpha': [1e-3, 1e-2, 0.05, 0.1, 0.3, 1]}, cv=inner, scoring=mse)),
 'Elastic Net': (lambda: ElasticNet(), lambda: GridSearchCV(ElasticNet(max_iter=20000), {'alpha': [1e-3, 1e-2, 0.1, 1], 'l1_ratio': [0.2, 0.5, 0.8]}, cv=inner, scoring=mse)),
 'PLS': (lambda: PLSRegression(), lambda: GridSearchCV(PLSRegression(), {'n_components': [1, 2, 3, 4, 5]}, cv=inner, scoring=mse)),
 'Random Forest': (lambda: RandomForestRegressor(random_state=SEED), lambda: GridSearchCV(RandomForestRegressor(n_estimators=100, random_state=SEED), {'max_depth': [None, 4], 'min_samples_leaf': [1, 3], 'max_features': [1.0, 'sqrt']}, cv=inner, scoring=mse)),
 'SVR': (lambda: SVR(), lambda: GridSearchCV(SVR(), {'C': [0.3, 1, 3, 10], 'epsilon': [0.05, 0.1, 0.2], 'gamma': ['scale', 0.01, 0.05]}, cv=inner, scoring=mse)),
 'GPR': (lambda: gpr(10), lambda: GPKernelSelect()),
}
X = P['X'][pool]; bc, lam = boxcox(P['df'][TFO].values[pool]); Z = StandardScaler().fit_transform(P['add'](X)); tt = {}
for name, (mk_def, mk_tun) in FAM.items():
    for cfg, mk in (('default', mk_def), ('tuned', mk_tun)):
        ts = []
        for _ in range(7):
            t = time.perf_counter(); mk().fit(Z, bc); ts.append(time.perf_counter() - t)
        tt[f'{name}|{cfg}'] = float(np.median(ts))
out['fit_time_s'] = tt; print('fit times (s):', {k: round(v, 4) for k, v in tt.items()}, flush=True)
drop = outliers_from_pool(P, 9); keep = np.array([i for i in range(len(pool)) if i not in set(drop)]); Xp = P['X'][pool][keep]; folds = fg.loo_folds(Xp, P['add'])
w = np.load('out/a4f_w1.npz'); pos = {int(s): i for i, s in enumerate(w['seeds'])}; res = {}
for T in (TFO, THX):
    raw = P['df'][T].values[pool][keep]; q10 = []; q1 = []
    for s in range(1, 201):
        y = raw[np.random.default_rng(s).permutation(len(raw))]; b, l = boxcox(y); yl = np.log(y); yp = bc_to_log(fg.loo_predict(folds, b, n_restarts=10), l)
        q10.append(1 - np.sum((yl - yp) ** 2) / np.sum((yl - yl.mean()) ** 2)); q1.append(float(w['q2_' + T][pos[s]]))
    q10 = np.array(q10); q1 = np.array(q1); d = q10 - q1
    res[T] = dict(n=200, mean_1=float(q1.mean()), mean_10=float(q10.mean()), max_1=float(q1.max()), max_10=float(q10.max()), mean_diff=float(d.mean()), max_abs_diff=float(np.abs(d).max()), n_diff_gt_0p01=int((np.abs(d) > 0.01).sum()))
    print(T, res[T], flush=True)
out['restarts'] = res; json.dump(out, open('out/a10_extra.json', 'w'), indent=1)


In [ ]:
%%writefile repro/bench/common.py
# Data loading, descriptor pipeline, Kennard-Stone split, model definition and metrics.
import copy, numpy as np, pandas as pd
from scipy.stats import boxcox, pearsonr, spearmanr, norm
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import LeaveOneOut
SEED=42
LOGF=['Half life_matrix_avg','tortuosity','interface_rot','matrix_CED','fiber_dia_or_average_particle_size_mm']
DROP=['hydrophyllic_gradient','matrix_sol_parameter','flory_huggins','logKOC_matrix_avg','sol_mismatch','logP_fiber','Lignin_Percentage','Cellulose_Percentage','Half life_fiber_avg','logKOC_fiber_avg','fiber_CED','fiber_rotatable_bond_count','Specific Interface Area']
PRE=[0,8,18]
MF=['Half life_matrix_avg','logP_matrix','matrix_CED']
FF=['Hemicellulose_Percentage','tortuosity','interface_rot','fiber_sol_parameter','fiber_dia_or_average_particle_size_mm']
TFO,THX='first_order_rate','hixson_rate'

def gpr(n_restarts=10):
    k=C(1.0,(1e-2,1e2))*RBF(1.0,(0.5,20.0))+WhiteKernel(0.1,(1e-2,1.0))
    return GaussianProcessRegressor(kernel=k,alpha=0.1,n_restarts_optimizer=n_restarts,normalize_y=True,random_state=SEED)

def bc_to_log(p,lam,eps=1e-10):
    if abs(lam)<0.01: return p
    return np.log(np.clip(np.power(np.clip(p*lam+1,eps,None),1.0/lam),eps,None))

def ks(X,n):
    d=cdist(X,X); i1,i2=np.unravel_index(np.argmax(d),d.shape); sel=[int(i1),int(i2)]
    rem=list(set(range(len(X)))-set(sel))
    while len(sel)<n:
        dd=cdist(X[rem],X[sel]); b=int(np.argmax(np.min(dd,axis=1))); sel.append(rem[b]); rem.remove(rem[b])
    return sel

def load(D):
    df=pd.read_excel(f'{D}/dataset.xlsx'); ad=pd.read_csv(f'{D}/applicability_domain.csv')
    feats_all=[c for c in df.columns if c not in (TFO,THX)]
    keep_rows=np.where(df[feats_all].notna().all(axis=1))[0]
    df=df.iloc[keep_rows].reset_index(drop=True); ad=ad.iloc[keep_rows].reset_index(drop=True)
    for f in LOGF:
        if (df[f]>0).all(): df[f]=np.log(df[f])
    df=df.drop(columns=DROP); feats=[f for f in feats_all if f in df.columns]
    m=np.ones(len(df),bool); m[PRE]=False
    df=df[m].reset_index(drop=True); ad=ad[m].reset_index(drop=True)
    X=df[feats].values.astype(float)
    ksi=ks(StandardScaler().fit_transform(X),9); kmask=np.zeros(len(X),bool); kmask[ksi]=True
    MI=[feats.index(f) for f in MF]; FI=[feats.index(f) for f in FF]
    add=lambda Z: np.hstack([Z]+[Z[:,a:a+1]*Z[:,b:b+1] for a in MI for b in FI])
    return dict(df=df,ad=ad,feats=feats,X=X,kmask=kmask,add=add)

def outliers_from_pool(P, n_out):
    if n_out==0: return []
    Xp=P['X'][~P['kmask']]; raw=P['df'][TFO].values[~P['kmask']]
    bc,lam=boxcox(raw); g=gpr(5); pr=np.zeros(len(raw))
    for tr,te in LeaveOneOut().split(Xp):
        sc=StandardScaler(); m=copy.deepcopy(g); m.fit(sc.fit_transform(P['add'](Xp[tr])),bc[tr])
        pr[te]=m.predict(sc.transform(P['add'](Xp[te]))).ravel()[0]
    r=np.log(raw)-bc_to_log(pr,lam); return np.argsort(np.abs(r/r.std()))[-n_out:].tolist()

def lin_ccc(y,yp):
    cov=np.cov(y,yp,ddof=0)[0,1]; return float(2*cov/(y.var()+yp.var()+(y.mean()-yp.mean())**2))

def metrics(y,yp):
    return dict(Q2=1-np.sum((y-yp)**2)/np.sum((y-y.mean())**2), CCC=lin_ccc(y,yp),
                r=pearsonr(y,yp)[0], rho=spearmanr(y,yp)[0], RMSE=float(np.sqrt(np.mean((y-yp)**2))),
                Bias=float(np.mean(yp-y)))

import re as _re
def study_key(r):
    s=str(r).strip(); s=_re.sub(r'\d+$','',s); m=_re.search(r'(19|20)\d\d',s); yr=m.group(0) if m else ''
    a=_re.split(r'[\s,(.]',s.strip())[0].lower().replace('el',''); return f'{a}_{yr}'


In [ ]:
%%writefile repro/bench/bench.py
# Benchmark of the seven model families with identical inputs, target and folds.
import sys, json, time, copy, numpy as np, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, sys.argv[2]); from common import *
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.gaussian_process.kernels import Matern
P=load(sys.argv[1]); pool=np.where(~P['kmask'])[0]
def gp_k(kind):
    base={'rbf':RBF(1.0,(0.5,20.0)),'m15':Matern(1.0,(0.5,20.0),nu=1.5),'m25':Matern(1.0,(0.5,20.0),nu=2.5)}[kind]
    return GaussianProcessRegressor(kernel=C(1.0,(1e-2,1e2))*base+WhiteKernel(0.1,(1e-2,1.0)),alpha=0.1,n_restarts_optimizer=0,normalize_y=True,random_state=SEED)
class GPKernelSelect:
    def fit(self,X,y):
        best=None
        for kd in ('rbf','m15','m25'):
            g=gp_k(kd).fit(X,y); lml=g.log_marginal_likelihood_value_
            if best is None or lml>best[0]: best=(lml,g)
        self.m=best[1]; return self
    def predict(self,X): return self.m.predict(X)
inner=KFold(5,shuffle=True,random_state=SEED)
FAM={
 'Ridge':(lambda: Ridge(), lambda: GridSearchCV(Ridge(),{'alpha':[0.01,0.1,1,10,100,1000]},cv=inner,scoring='neg_mean_squared_error')),
 'Lasso':(lambda: Lasso(), lambda: GridSearchCV(Lasso(max_iter=20000),{'alpha':[1e-3,1e-2,0.05,0.1,0.3,1]},cv=inner,scoring='neg_mean_squared_error')),
 'Elastic Net':(lambda: ElasticNet(), lambda: GridSearchCV(ElasticNet(max_iter=20000),{'alpha':[1e-3,1e-2,0.1,1],'l1_ratio':[0.2,0.5,0.8]},cv=inner,scoring='neg_mean_squared_error')),
 'PLS':(lambda: PLSRegression(), lambda: GridSearchCV(PLSRegression(),{'n_components':[1,2,3,4,5]},cv=inner,scoring='neg_mean_squared_error')),
 'Random Forest':(lambda: RandomForestRegressor(random_state=SEED), lambda: GridSearchCV(RandomForestRegressor(n_estimators=100,random_state=SEED),{'max_depth':[None,4],'min_samples_leaf':[1,3],'max_features':[1.0,'sqrt']},cv=inner,scoring='neg_mean_squared_error')),
 'SVR':(lambda: SVR(), lambda: GridSearchCV(SVR(),{'C':[0.3,1,3,10],'epsilon':[0.05,0.1,0.2],'gamma':['scale',0.01,0.05]},cv=inner,scoring='neg_mean_squared_error')),
 'GPR':(lambda: gpr(3), lambda: GPKernelSelect()),
}
res={}
for n_out in (0,9):
    drop=outliers_from_pool(P,n_out); keep=np.array([i for i in range(len(pool)) if i not in set(drop)]); idx=pool[keep]; X=P['X'][idx]
    res[n_out]={}
    for T in (TFO,THX):
        raw=P['df'][T].values[idx]; ly=np.log(raw); bc,lam=boxcox(raw); res[n_out][T]={}
        for name,(mk_def,mk_tun) in FAM.items():
            for cfg,mk in (('default',mk_def),('tuned',mk_tun)):
                pr=np.zeros(len(bc))
                for a,b in LeaveOneOut().split(X):
                    sc=StandardScaler(); m=mk(); m.fit(sc.fit_transform(P['add'](X[a])),bc[a]); pr[b]=np.ravel(m.predict(sc.transform(P['add'](X[b]))))[0]
                loo=metrics(ly,bc_to_log(pr,lam))
                sc=StandardScaler(); Z=sc.fit_transform(P['add'](X)); m=mk(); m.fit(Z,bc); r2in=1-np.sum((ly-bc_to_log(np.ravel(m.predict(Z)),lam))**2)/np.sum((ly-ly.mean())**2)
                res[n_out][T][f'{name}|{cfg}']=dict(R2=float(r2in),**{k:float(v) for k,v in loo.items()})
                print(f"n={len(bc)} {T[:5]} {name:<13} {cfg:<7} R2={r2in:.3f} Q2={loo['Q2']:.3f} CCC={loo['CCC']:.3f} rho={loo['rho']:.3f} RMSE={loo['RMSE']:.3f}",flush=True)
        mats=P['ad']['Matrix'].values[idx]; lp=P['df']['logP_matrix'].values[idx]; Xb=X
        base={}
        for nm in ('null','OLS (8 base)','logP only','logP quintile','matrix-class lookup'):
            pr=np.zeros(len(bc))
            for a,b in LeaveOneOut().split(Xb):
                if nm=='null': pr[b]=bc[a].mean()
                elif nm=='OLS (8 base)': sc=StandardScaler(); pr[b]=LinearRegression().fit(sc.fit_transform(Xb[a]),bc[a]).predict(sc.transform(Xb[b]))
                elif nm=='logP only': pr[b]=LinearRegression().fit(lp[a,None],bc[a]).predict(lp[b,None])
                elif nm=='logP quintile':
                    edges=np.quantile(lp[a],[0.2,0.4,0.6,0.8]); qa=np.digitize(lp[a],edges); qb=np.digitize(lp[b],edges)[0]; sel=bc[a][qa==qb]; pr[b]=sel.mean() if len(sel) else bc[a].mean()
                else:
                    sel=bc[a][mats[a]==mats[b][0]]; pr[b]=sel.mean() if len(sel) else bc[a].mean()
            mm=metrics(ly,bc_to_log(pr,lam)); base[nm]={k:float(v) for k,v in mm.items()}
            print(f"n={len(bc)} {T[:5]} baseline {nm:<20} Q2={mm['Q2']:.3f} CCC={mm['CCC']:.3f}",flush=True)
        res[n_out][T]['_baselines']=base
X=P['X'][pool]; bc,lam=boxcox(P['df'][TFO].values[pool]); Z=StandardScaler().fit_transform(P['add'](X)); tt={}
for name,(mk_def,mk_tun) in FAM.items():
    for cfg,mk in (('default',mk_def),('tuned',mk_tun)):
        ts=[]
        for _ in range(3):
            t=time.perf_counter(); mk().fit(Z,bc); ts.append(time.perf_counter()-t)
        tt[f'{name}|{cfg}']=float(np.median(ts))
res['fit_time_s_n49']=tt; print('fit times (s):',{k:round(v,4) for k,v in tt.items()})
json.dump({str(k):v for k,v in res.items()},open(sys.argv[3],'w'),indent=1)


In [ ]:
%%writefile repro/bench/shap_nolog.py
# Shapley ranking with and without the log transformation of the skewed descriptors.
import sys, json, itertools, math, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, sys.argv[2]); from common import *
D=sys.argv[1]; P=load(D); feats=P['feats']; pool=np.where(~P['kmask'])[0]
df=pd.read_excel(f'{D}/dataset.xlsx'); fa=[c for c in df.columns if c not in (TFO,THX)]
rows=np.where(df[fa].notna().all(axis=1))[0]; df=df.iloc[rows].reset_index(drop=True)
m=np.ones(len(df),bool); m[PRE]=False; df=df[m].reset_index(drop=True)
Xraw=df[feats].values.astype(float)
assert np.allclose(np.log(Xraw[:,feats.index('tortuosity')]),P['X'][:,feats.index('tortuosity')])
Mf=len(feats); subsets=list(itertools.product([0,1],repeat=Mf)); S=np.array(subsets,bool); key={s:i for i,s in enumerate(subsets)}
G={'matrix chemistry':['Half life_matrix_avg','logP_matrix','matrix_CED'],'fibre geometry':['tortuosity','fiber_dia_or_average_particle_size_mm'],
   'fibre chemistry':['Hemicellulose_Percentage','fiber_sol_parameter'],'interface_rot':['interface_rot']}
def values(model,sc,X,bg):
    out=np.zeros((len(X),len(S)))
    for i,x in enumerate(X):
        rows=np.where(S[:,None,:],x[None,None,:],bg[None,:,:]).reshape(-1,Mf)
        out[i]=model.predict(sc.transform(P['add'](rows))).reshape(len(S),len(bg)).mean(1)
    return out
def shap_from_v(V,players):
    K=len(players); ph=np.zeros((V.shape[0],K))
    for g in range(K):
        for c in itertools.product([0,1],repeat=K):
            if c[g]: continue
            z=sum(c); c2=list(c); c2[g]=1
            def idx(cc):
                mask=[0]*Mf
                for h in range(K):
                    if cc[h]:
                        for j in players[h]: mask[j]=1
                return key[tuple(mask)]
            w=math.factorial(z)*math.factorial(K-z-1)/math.factorial(K); ph[:,g]+=w*(V[:,idx(c2)]-V[:,idx(c)])
    return ph
out={}
for n_out in (0,9):
    drop=outliers_from_pool(P,n_out); keep=np.array([i for i in range(len(pool)) if i not in set(drop)]); idx=pool[keep]
    out[n_out]={}
    for lab,X in (('log',P['X'][idx]),('nolog',Xraw[idx])):
        for T in (TFO,THX):
            raw=P['df'][T].values[idx]; bc,lam=boxcox(raw); sc=StandardScaler().fit(P['add'](X)); g=gpr(3).fit(sc.transform(P['add'](X)),bc)
            V=values(g,sc,X,X)
            ind=np.abs(shap_from_v(V,[[j] for j in range(Mf)])).mean(0); ind=ind/ind.sum(); oi=np.argsort(-ind)
            grp_players=[[feats.index(f) for f in G[k]] for k in G]; gr=np.abs(shap_from_v(V,grp_players)).mean(0); gr=gr/gr.sum(); og=np.argsort(-gr)
            out[n_out][f'{lab}|{T}']=dict(order=[feats[j] for j in oi],share=[float(ind[j]) for j in oi],group_order=[list(G)[j] for j in og],group_share=[float(gr[j]) for j in og])
            print(len(idx),lab,T[:5],'| top4:',', '.join(f'{feats[j]} {ind[j]:.3f}' for j in oi[:4]),'| groups:',', '.join(f'{list(G)[j]} {gr[j]:.3f}' for j in og),flush=True)
json.dump({str(k):v for k,v in out.items()},open(sys.argv[3],'w'),indent=1)


In [ ]:
%%writefile repro/an/collect_values.py
# Collects the reported values from the result files.
import json, glob, numpy as np
AN = 'out'; B3 = '../bench/out'
A1 = json.load(open(f'{AN}/a1_basic.json')); A7 = json.load(open(f'{AN}/a7_misc.json')); SH = json.load(open(f'{AN}/a6_shap_final40_b1000.json')); BE = json.load(open(f'{B3}/bench.json'))
FO, HC = 'first_order_rate', 'hixson_rate'
def f(x, d=3):
    s = f'{x:.{d}f}'
    if float(s) == 0: s = s.lstrip('-')
    return s.replace('-', '−')
N = {}
for tag, key in (('f', 'final40'), ('p', 'pool49')):
    for T, t in ((FO, 'fo'), (HC, 'hc')):
        o = A1[key][T]
        for k in ('Q2', 'CCC', 'r', 'rho', 'RMSE', 'Bias'): N[f'{tag}_{k}_{t}'] = o['loo'][k]
        for k in ('r2_ext', 'Q2F1', 'Q2F2', 'Q2F3', 'CCC_ext', 'RMSEP', 'rho_ext', 'sd_obs', 'sd_pred'): N[f'{tag}_{k}_{t}'] = o['ext'][k]
        N[f'{tag}_R2in_{t}'] = o['R2_in']; N[f'{tag}_lam_{t}'] = o['lam']
        g = o['GT']; N[f'{tag}_k_{t}'] = g['k']; N[f'{tag}_kp_{t}'] = g['kp']; N[f'{tag}_c3_{t}'] = min(g['c3a'], g['c3b']); N[f'{tag}_c5_{t}'] = g['c5']; N[f'{tag}_R0_{t}'] = g['R0']; N[f'{tag}_R0p_{t}'] = g['R0p']
        c = o['calII']; N[f'{tag}_L1_{t}'] = c['L1_19']; N[f'{tag}_SQ10_{t}'] = c['SQ_cdf10']; N[f'{tag}_L1c10_{t}'] = c['L1_cdf10']; N[f'{tag}_L1c19_{t}'] = c['L1_cdf19']; N[f'{tag}_SQ19_{t}'] = c['SQ_cdf19']
        for lv in (50, 68, 90, 95): N[f'{tag}_cov{lv}_{t}'] = c[f'cov{lv}']['k']
        N[f'{tag}_wil_{t}'] = c['cov95']['wilson']; N[f'{tag}_n'] = o['n']
names = SH[FO]['single']['players']
for T, t in ((FO, 'fo'), (HC, 'hc')):
    s = SH[T]['single']['boot1000']
    for nm, short in (('logP_matrix', 'logp'), ('tortuosity', 'tort'), ('interface_rot', 'irot'), ('hemicellulose', 'hemi'), ('matrix CED', 'ced'), ('fibre diameter', 'dia'), ('matrix half-life', 'hl'), ('fibre sol. parameter', 'fsol')):
        i = names.index(nm); N[f'b1_{short}_{t}'] = s['top1'][i]; N[f'b3_{short}_{t}'] = s['top3'][i]
    N[f'bH2_{t}'] = SH[T]['H2']['boot1000']['top1'][0]; N[f'bM3_{t}'] = SH[T]['M3']['boot1000']['top1'][0]
    b = A7['base_' + T]; N[f'look_q2_{t}'] = b['q2_lookup']; N[f'quin_q2_{t}'] = b['q2_quintile']; N[f'gpr_gt_look_{t}'] = b['gpr_better_than_lookup']; N[f'gpr_gt_quin_{t}'] = b['gpr_better_than_quintile']
    N[f'extcov_{t}'] = A7['extcov_' + T]['inside95']; N[f'mge3_ccc_{t}'] = A7['matrix_ge3'][T]['CCC']; N[f'mge3_q2_{t}'] = A7['matrix_ge3'][T]['Q2']
    N[f'irot_rho_{t}'] = A7['dir_' + T]['interface_rot']['rho_value_vs_shap']
N['mge3_n'] = A7['matrix_ge3']['n']; N['zmin'] = min(abs(d['z']) for d in A7['pool_resid'][:9]); N['zmax'] = max(abs(d['z']) for d in A7['pool_resid'][:9])
import sys as _sys, warnings as _w; _w.filterwarnings('ignore'); _sys.path.insert(0, '.')
from common import load as _load
_P = _load('data'); _ad = _P['ad']; _pool = np.where(~_P['kmask'])[0]; _exc = {e[0] for e in A1['excluded9']}
_canon = lambda m: 'PBS' if m in ('PBSu', 'Bio-PBS', 'PBS') else m
_m = np.array([_canon(m) for m in _ad['Matrix'].values]); _nm = _ad['Sample Name'].astype(str).values; _cnt = {m: int((_m[_pool] == m).sum()) for m in set(_m[_pool])}
_rare = [i for i in _pool if _cnt[_m[i]] <= 2]; _com = [i for i in _pool if _cnt[_m[i]] > 2]
N['rare_n'] = len(_rare); N['rare_exc'] = sum(_nm[i] in _exc for i in _rare); N['com_n'] = len(_com); N['com_exc'] = sum(_nm[i] in _exc for i in _com)
assert (N['rare_n'], N['rare_exc'], N['com_n'], N['com_exc']) == (9, 6, 40, 3)
EXC9 = [dict(sample=e[0], matrix=e[1], filler=e[2], ref=e[3]) for e in A1['excluded9']]
zs = [np.load(p) for p in sorted(glob.glob(f'{AN}/a4f_w*.npz'))]
if zs:
    for T, t in ((FO, 'fo'), (HC, 'hc')):
        q = np.concatenate([z['q2_' + T] for z in zs]); N[f'yr_n'] = len(q); N[f'yr_mean_{t}'] = float(q.mean()); N[f'yr_sd_{t}'] = float(q.std()); N[f'yr_max_{t}'] = float(q.max()); N[f'yr_b_{t}'] = int(np.sum(q >= N[f'f_Q2_{t}']))
        N[f'yr_p95_{t}'] = float(np.percentile(q, 95))
try:
    S = json.load(open(f'{AN}/a8_selperm.json')); N.update({f'sp_{k}': v for k, v in S.items()})
except Exception: pass
def bench(n, T):
    rows = {}
    for k, v in BE[str(n)][T].items():
        if k.startswith('_'): continue
        fam, cfg = k.split('|'); rows.setdefault(fam, {})[cfg] = v
    return rows
if __name__ == '__main__':
    for k in sorted(N): print(k, N[k] if not isinstance(N[k], float) else round(N[k], 4))


## 3. Run

In [ ]:
# final model, sample accounting, baselines, sensitivity to isolated composites
run(['a1_basic.py'], AN, ['out/a1_basic.json'], 'out/a1_basic.log')
run(['a2_accounting.py'], AN, ['out/a2_accounting.json'], 'out/a2_accounting.log')
run(['a7_misc.py'], AN, ['out/a7_misc.json'], 'out/a7_misc.log')
run(['a9_prelim.py'], AN, ['out/a9_prelim.json'], 'out/a9_prelim.log')

In [ ]:
# benchmark of the model families; Shapley ranking with and without log transformation
run(['bench.py', DATA, BN, 'out/bench.json'], BN, ['out/bench.json'], 'out/bench.log')
run(['shap_nolog.py', DATA, BN, 'out/shap_nolog.json'], BN, ['out/shap_nolog.json'], 'out/shap_nolog.log')

In [ ]:
# bootstrap of the Shapley ranking, 1,000 resamples
run(['a6_shap.py', 'final40', '1000', 'out/a6_shap_final40_b1000.json'], AN, ['out/a6_shap_final40_b1000.json'], 'out/a6_final40_b1000.log')

In [ ]:
# y-randomisation, 10,000 permutations on two cores (about 70 min)
run_parallel([(['a4f_yrand_fast.py', '1', '5000', 'out/a4f_w1'], ['out/a4f_w1.json'], 'out/a4f_w1.log'),
              (['a4f_yrand_fast.py', '5001', '10000', 'out/a4f_w2'], ['out/a4f_w2.json'], 'out/a4f_w2.log')], AN)

In [ ]:
# permutation test including the outlier screening, 1,200 permutations (about 30 min)
run(['a8_selperm.py', '1', '1200', 'out/a8_w1'], AN, ['out/a8_w1_obs.json'], 'out/a8_w1.log')
run(['a8_merge.py', '1200'], AN, ['out/a8_selperm.json'], 'out/a8_merge.log')

In [ ]:
# fit times and optimiser-restart check (about 20 min)
run(['a10_extra.py'], AN, ['out/a10_extra.json'], 'out/a10_extra.log')

## 4. Results

In [ ]:
home = os.getcwd(); os.chdir(AN); ns = {}
try: exec(compile(open('collect_values.py').read(), 'collect_values.py', 'exec'), ns)
finally: os.chdir(home)
N = {k: (list(v) if isinstance(v, tuple) else v) for k, v in ns['N'].items()}
json.dump(N, open(os.path.join(BASE, 'reported_values.json'), 'w'), indent=1)
for k in sorted(N): print('%-22s %s' % (k, '%.5f' % N[k] if isinstance(N[k], float) else N[k]))

In [ ]:
# comparison with the reference results (folder "reference"), tolerance 0.0005
def leaves(x, p=''):
    if isinstance(x, dict):
        for k in sorted(x, key=str): yield from leaves(x[k], p + '/' + str(k))
    elif isinstance(x, (list, tuple)):
        for i, v in enumerate(x): yield from leaves(v, p + '/' + str(i))
    else: yield p, x

def compare(name, got, ref, skip=()):
    g = dict(leaves(got)); bad = []; n = 0
    for k, v in leaves(ref):
        if any(s in k for s in skip): continue
        n += 1
        if k not in g or (g[k] != v if isinstance(v, (bool, str, type(None))) else abs(float(g[k]) - float(v)) > 5e-4): bad.append((k, g.get(k), v))
    print('%-28s %6d values  %s' % (name, n, 'match' if not bad else '%d differ: %s' % (len(bad), bad[:5]))); return not bad

if os.path.isdir('reference'):
    ok = compare('reported_values.json', N, json.load(open('reference/reported_values.json')), ('obs_check',))
    for f, d, skip in (('a1_basic.json', AN, ()), ('a2_accounting.json', AN, ()), ('a7_misc.json', AN, ()), ('a9_prelim.json', AN, ()), ('a6_shap_final40_b1000.json', AN, ('time', 'sec')),
                       ('a8_selperm.json', AN, ('obs_check',)), ('a10_extra.json', AN, ('fit_time',)), ('bench.json', BN, ('fit_time',)), ('shap_nolog.json', BN, ())):
        ok &= compare(f, json.load(open(os.path.join(d, 'out', f))), json.load(open(os.path.join('reference', f))), skip)
    for w in ('a4f_w1.npz', 'a4f_w2.npz'):
        a = np.load(os.path.join(AN, 'out', w)); b = np.load(os.path.join('reference', w)); ia = np.argsort(a['seeds']); ib = np.argsort(b['seeds'])
        same = np.array_equal(a['seeds'][ia], b['seeds'][ib]) and all(np.abs(a[k][ia] - b[k][ib]).max() <= 5e-4 for k in a.files if k != 'seeds')
        print('%-28s %6d permutations  %s' % (w, len(ia), 'match' if same else 'differ')); ok &= bool(same)
    print('All values reproduced.' if ok else 'Differences found.')